# Table 5: **Baseline**

## Objective
In this experiment, we compare the performance of our proposed benchmark (Multi-Sample Consensus) against several models across different games. This experiment aims to show that the benchmark is consistent across games and is useful as a yardstick against models. This is referred in our paper as **Experiment 5**.

---

## Methodology
- **Models Used in Our Experiment:**  
  - GPT-4o Mini  
  - Qwen2.5-72B (int4)  
  - Mistral-Small  

- **Evaluation Metrics:**  
  The script computes the following metrics for each model:  
  - % 5/6-way agreement  
  - % 6-way agreement  
  - % Any agreement  

---

## Results
The results of this experiment demonstrate the reliability of our proposed benchmark across different games. These findings are presented in **Table 5** of our paper.

In [1]:
import os
import eval_utils as evaluation
import json
import pandas as pd
from IPython.display import display

raw_path = '../our_games_descriptions/base/output/baselines'

variants = ['base', 'random_approach', 'centered_approach', 'prev_prox_approach', 'multi_consensus_alpha1', 'freq_restrict_multi_consensus_alpha1', 'two_phase_approach_alpha1']

results = {}
ISSUES_NUM = 5
AGENTS_NUM = 6

for variant in variants:

    if variant == 'base':
        directory = '../our_games_descriptions/base/output/original_code/gpt4o-mini' # We use GPT-4o mini as a reference point
    else:
        directory = os.path.join(raw_path, variant)

    agents, role_to_agents, incentive_to_agents = evaluation.load_setup(directory, AGENTS_NUM, num_issues=ISSUES_NUM)
    answers_files = [ os.path.join(directory,filename) for filename in os.listdir(directory) if filename.startswith("history")]

    num_rounds = 0
    for file_ in answers_files:
        answers = json.load(open(file_))
        _num_rounds = len(answers['rounds'])
        num_rounds = max(num_rounds, _num_rounds)

    # Track statistics
    feasible_in_last_step = 0
    accepted_by_all_in_last_step = 0
    contained_feasible_deal = 0
    successfull_games = 0
    total_rounds = 0

    # Loop through all answer files (each represents a game)
    for file_ in answers_files:
        answers = json.load(open(file_))
        
        if len(answers['rounds']) != num_rounds:
            print(f"WARNING: Game {file_} has a different number of rounds")
            continue
        total_rounds += len(answers['rounds'])
        successfull_games += 1

        # Extract deals for this game
        feasible_found = False

        # Extract the name of the first player (p1) to validate feasibility throughout the game
        p1_name = answers['rounds'][0]['agent']

        total_deals = 0
        
        for i, round_ in enumerate(answers['rounds']):
            name, answer = round_['agent'], round_['public_answer']
            deal_unformatted, issues_suggested = evaluation.extract_deal(answer, ISSUES_NUM)

            try:
                deal = evaluation.format_deal(deal_unformatted, ISSUES_NUM)
            except:
                print(f"Error in game {file_} round {i}")
                continue

            if issues_suggested >= ISSUES_NUM:
                total_deals += 1

            # Check if the deal was feasible at any point (Deal must have been proposed by p1)
            if evaluation.is_feasible(agents, deal) and name == p1_name:
                feasible_found = True
        

        # CHECK GAME COMPLETION METRICS

        last_deal = evaluation.format_deal(evaluation.extract_deal(answers['rounds'][-1]['public_answer'], ISSUES_NUM)[0], ISSUES_NUM)
        
        # 1. Check if the last deal is feasible
        if evaluation.is_feasible(agents, last_deal):
            feasible_in_last_step += 1

        # 2. Check if the last deal is acceptable by all agents
        all_accept = all(evaluation.calculator(agents[agent]["scores"], last_deal, ISSUES_NUM, verbose=False) >= agents[agent]["scores"]["min"] for agent in agents)
        if all_accept:
            accepted_by_all_in_last_step += 1

        # 3. Check if any deal during the game was in the feasibility set
        if feasible_found:
            contained_feasible_deal += 1

    # Compute percentages
    num_games = successfull_games
    perc_feasible_last = (feasible_in_last_step / num_games) * 100
    perc_accepted_all_last = (accepted_by_all_in_last_step / num_games) * 100
    perc_feasible_any = (contained_feasible_deal / num_games) * 100

    results[variant] = {
        '5/6-way (%)': perc_feasible_last,
        '6-way (%)': perc_accepted_all_last,
        'Any (%)': perc_feasible_any,
    }

df = pd.DataFrame(results).T
display(df)

,5/6-way (%),6-way (%),Any (%)
base,55.0,5.0,90.0
random_approach,18.0,6.0,25.0
centered_approach,100.0,0.0,100.0
prev_prox_approach,18.0,4.0,19.0
multi_consensus_alpha1,50.0,19.0,66.0
freq_restrict_multi_consensus_alpha1,29.0,0.0,29.0
two_phase_approach_alpha1,68.0,0.0,68.0
